### Agent

<small>User prompt: "Schedule a meeting with Joe next week and send him a confirmation email."

To complete this, the LLM with help of Agent will do the following tasks:  
- Check your calendar
- Find free time slots
- Check Joe’s availability
- Create the calendar event
- Send a confirmation email
- Ask follow-up questions if information is missing</small>

### Chain vs Agent

<small>

**A Chain** is a fixed sequence of LLM calls. You define every step in code before the task runs.  
**An Agent** decides dynamically which steps to take based on what it observes at runtime.

| | Chain | Agent |
|---|---|---|
| **Steps** | Fixed, defined upfront | Dynamic, chosen by the LLM |
| **Decision maker** | You (the programmer) | The LLM |
| **Tools** | Called in a fixed order | Called based on need |
| **Best for** | Predictable pipelines | Open-ended, conditional tasks |

---

**Finance example**
- *Chain*: "Summarise this earnings report" → fixed prompt → fixed output. You already know the path.
- *Agent*: "Is my portfolio overexposed to tech?" → checks current holdings → checks sector weights → checks risk threshold → re-plans if data is missing → proposes rebalancing. The path depends on what each step returns.

**Education example**
- *Chain*: "Grade this essay against this rubric" → rubric + essay → score. One step, predictable.
- *Agent*: "What courses should I take next semester?" → checks transcript → verifies prerequisites → detects schedule conflicts → ranks options. Cannot plan the sequence until step 1 completes.

**Healthcare example**
- *Chain*: "Summarise this lab report for the patient" → template + report → output. Fixed path.
- *Agent*: "Book the earliest available cardiology appointment" → checks doctor availability → checks patient calendar → handles conflicts → confirms booking. Each step depends on the previous result.

</small>

**Agent Loop**  
<small>It repeatedly performs:  
- Reason  
- Choose an action  
- Call a tool  
- Observe result  
- Update plan  
- Repeat until done</nsmall>

#### When to Use Agents vs Chains vs Direct Calls

<small>

| Scenario | Right tool | Why |
|---|---|---|
| Translate a document | Direct API call | Single step, no decisions needed |
| Classify customer support emails | Chain | Fixed input → label, predictable path |
| Answer a product FAQ | Chain (RAG) | Retrieve + generate, known structure |
| "Plan my investment strategy" | Agent | Unknown number of steps, conditional decisions |
| "Enrol me in the best ML course for my level" | Agent | Must check level, prerequisites, seat availability |
| "Monitor portfolio and alert on drops > 5%" | Agent + loop | Ongoing observation + conditional action |

---

**Agents are overkill when:**
- The path from input to output never changes
- There are fewer than 3 steps and no branching
- The task does not depend on what earlier steps returned
- Latency or cost is a hard constraint

**Finance overkill**: Converting USD to EUR at a fixed rate — a single function call, not an agent.  
**Education overkill**: Marking a multiple-choice quiz — deterministic, no tool selection needed.

**Rule of thumb:**  
If you can draw the complete flowchart *before* running the task, use a chain.  
If the flowchart depends on what the agent discovers, use an agent.

</small>

### ReAct pattern(Reason + Act)

```mermaid
---
title: ReAct Pattern
---
flowchart LR
    A[User Query] --> B(REason \n Analyze & Decide)
    B --> C>Task Complete?]
    C -->|Yes| D[Final Result]
    C -->|No| E[ACTion \nUse Tools & Execute]
    E --> F[Observation \n review result]
    F --> B
```

<small>ReAct pattern combines:  
- Reasoning steps (“Thought”)  
- Actions (“Tool Calls”)  
- Observations (“Tool Results”)  

**Typical Flow**:  

Thought: I need weather data  
Action: call_weather_api("Bangalore")  
Observation: Rainy, 24°C  

Thought: Need calendar availability  
Action: call_calendar_api()  
Observation: Free at 4 PM  

Thought: Enough information gathered  
Final Answer: Schedule outdoor meeting tomorrow instead  

The model alternates between:  
Internal reasoning  
External actions</nsmall>

### Tools

<small>Tools are external functions/APIs the agent can use.  
Examples:  
Weather API  
Calendar API   
Search engine   
Email sender</nsmall>

<small>How LLM selects the appropriate tool?  
The LLM reads:   
- User request  
- Available tool descriptions  
- Previous observations  

Then predicts: Which tool best helps accomplish the task?</nsmall>

<small>Available Tools
| Tool           | Purpose               |
| -------------- | --------------------- |
| `search_web`   | Search internet       |
| `send_email`   | Send email            |
| `create_event` | Create calendar event |</small>


#### Domain-specific Tool Schemas

<small>The same JSON schema structure applies across all domains.  
Notice how the `description` field is the primary signal the LLM uses to decide *when* to call each tool.  
High-stakes tools (booking, trading, enrolment) should flag this in their description so the agent can apply an approval gate.</small>

In [ ]:
import json

# Finance: portfolio lookup
portfolio_schema = {
    "name": "get_portfolio",
    "description": "Return current holdings and sector weights for a user's investment portfolio.",
    "parameters": {
        "type": "object",
        "properties": {
            "user_id": {"type": "string", "description": "Unique investor account ID"},
            "include_performance": {
                "type": "boolean",
                "description": "Include YTD return figures",
                "default": False,
            },
        },
        "required": ["user_id"],
    },
}

# Education: course search
course_search_schema = {
    "name": "search_courses",
    "description": "Search available courses by topic, difficulty level, or prerequisite.",
    "parameters": {
        "type": "object",
        "properties": {
            "topic": {"type": "string", "description": "Subject area, e.g. 'machine learning'"},
            "level": {
                "type": "string",
                "enum": ["beginner", "intermediate", "advanced"],
            },
            "prerequisites_met": {
                "type": "array",
                "items": {"type": "string"},
                "description": "List of course codes the student has already completed",
            },
        },
        "required": ["topic"],
    },
}

# Healthcare: appointment booking (high-stakes — requires approval)
appointment_schema = {
    "name": "book_appointment",
    "description": "Book a medical appointment. HIGH-STAKES: requires human approval before execution.",
    "parameters": {
        "type": "object",
        "properties": {
            "patient_id": {"type": "string"},
            "doctor_specialty": {"type": "string", "description": "e.g. 'cardiology', 'general'"},
            "preferred_date": {"type": "string", "format": "date", "description": "ISO 8601 date"},
            "reason": {"type": "string", "description": "Brief clinical reason for visit"},
        },
        "required": ["patient_id", "doctor_specialty"],
    },
}

# E-commerce: order status lookup
order_status_schema = {
    "name": "get_order_status",
    "description": "Retrieve current status and estimated delivery for a customer order.",
    "parameters": {
        "type": "object",
        "properties": {
            "order_id": {"type": "string"},
            "include_history": {"type": "boolean", "default": False},
        },
        "required": ["order_id"],
    },
}

for schema in [portfolio_schema, course_search_schema, appointment_schema, order_status_schema]:
    print(f"\n{'='*50}")
    print(f"Domain: {schema['name']}")
    print(json.dumps(schema, indent=2))

JSON Tool Schema

In [1]:
import json

send_email_schema = {
    "name": "send_email",
    "description": "Send an email to a recipient",
    "parameters": {
        "type": "object",
        "properties": {
            "to": {"type": "string"},
            "subject": {"type": "string"},
            "body": {"type": "string"},
        },
        "required": ["to", "subject", "body"],
    },
}

print(json.dumps(send_email_schema, indent=2))

{
  "name": "send_email",
  "description": "Send an email to a recipient",
  "parameters": {
    "type": "object",
    "properties": {
      "to": {
        "type": "string"
      },
      "subject": {
        "type": "string"
      },
      "body": {
        "type": "string"
      }
    },
    "required": [
      "to",
      "subject",
      "body"
    ]
  }
}


<small>This JSON schema tells the model exactly which fields the tool expects.  
The required keys are `to`, `subject`, and `body`, so the agent can validate arguments before calling the tool.</small>

<small> Using the available tools and their tool schemas the model decides the tools that are required and passes the appropriate arguments.  
send_email(  
&emsp;to="joe@example.com",  
&emsp;subject="Meeting Confirmed",  
&emsp;body="Your meeting is confirmed."  
)</small>

Tool creation and management

In [2]:
import os
import sys
sys.path.append('Track2/Functions')

from C2_3_Func import Tool, WeatherTool, ExchangeRateTool, NewsTool, TimezoneTool, EmailTool, execute_tool

In [3]:
# WeatherTool is defined in C2_3_Func and imported above

In [ ]:
# ExchangeRateTool is defined in C2_3_Func and imported above

In [ ]:
# NewsTool is defined in C2_3_Func and imported above

In [ ]:
# TimezoneTool is defined in C2_3_Func and imported above

In [ ]:
# EmailTool is defined in C2_3_Func and imported above

In [ ]:
# Load API key
WEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
EXCHANGE_API_KEY = os.getenv("EXCHANGERATE_API_KEY")
NEWS_API_KEY = os.getenv("NEWSAPI_API_KEY")
NINJAS_API_KEY = os.getenv("APININJAS_API_KEY")
RESEND_API_KEY = os.getenv("RESEND_API_KEY")
# Tools registry
TOOLS = {
    "weather": WeatherTool(WEATHER_API_KEY),
    "exchange_rate": ExchangeRateTool(EXCHANGE_API_KEY),
    "news": NewsTool(NEWS_API_KEY),
    "timezone": TimezoneTool(NINJAS_API_KEY),
    "email": EmailTool(RESEND_API_KEY)
}

In [ ]:
# execute_tool is defined in C2_3_Func and imported above

In [ ]:
result = execute_tool(
    "weather",
    TOOLS,
    city="Bangalore"
)
print(result)

In [ ]:
result = execute_tool(
    "exchange_rate",
    TOOLS,
    base_currency="USD",
    target_currency="INR"
)
print(result)

In [ ]:
result = execute_tool(
    "news",
    TOOLS,
    topic="Stocks",
    limit=3
)
print(result)

In [ ]:
result = execute_tool(
    "timezone",
    TOOLS,
    timezone="Europe/London"
)
print(result)

In [ ]:
result = execute_tool(
    "email",
    TOOLS,
    to="ogiboyinaakash@gmail.com",
    subject="Hello",
    body="This email was sent by my AI agent."
)
print(result)

### Multi-Step Task Execution

<small>Example task:  
"Book a flight and add it to my calendar."  

The agent may need:  
- Search flights  
- Compare prices  
- Choose best option  
- Book ticket  
- Generate confirmation  
- Add calendar event  
- Notify user  

This requires:  
- State tracking  
- Iterative planning  
- Tool orchestration  
- Failure handling </nsmall>

<small>**Planning Loop**  
while not task_completed:  
&emsp;think()  
&emsp;choose_action()  
&emsp;execute_tool()  
&emsp;observe_result()  
&emsp;update_plan()</nsmall>

#### Domain-specific Multi-Step Scenarios

<small>

**Finance: "Rebalance my portfolio to reduce tech exposure"**

The agent cannot know step 4 until it has executed step 3:

| Step | Action | Tool |
|---|---|---|
| 1 | Fetch current holdings | `get_portfolio` |
| 2 | Classify each holding by sector | `classify_sectors` |
| 3 | Identify overweight sectors (tech > 40%) | Reasoning only |
| 4 | Rank liquid positions for partial sale | `rank_by_liquidity` |
| 5 | **Request human approval** — "Sell 15% of NVDA?" | Approval gate |
| 6 | Execute trade if approved | `execute_order` |
| 7 | Log all decisions with timestamps | Decision log |

---

**Education: "Enrol me in the best Data Science track for my background"**

| Step | Action | Tool |
|---|---|---|
| 1 | Retrieve completed courses | `get_transcript` |
| 2 | Search available DS courses | `search_courses` |
| 3 | Check prerequisites for each candidate | `check_prerequisites` |
| 4 | Detect schedule conflicts | `check_schedule` |
| 5 | Rank by fit score | Reasoning only |
| 6 | **Request human confirmation** — "Enrol in ML-301?" | Approval gate |
| 7 | Submit enrolment + add to calendar | `enrol` + `create_event` |

---

**Healthcare: "Book the earliest cardiology slot that fits my schedule"**

| Step | Action | Tool |
|---|---|---|
| 1 | Retrieve patient availability | `get_patient_calendar` |
| 2 | Search open cardiology slots | `search_appointments` |
| 3 | Find earliest overlap | Reasoning only |
| 4 | **Require confirmation** — show proposed slot to patient | Approval gate |
| 5 | Confirm booking + send reminder | `book_appointment` + `send_reminder` |

---

**Key insight:** In all three cases the agent cannot write the flowchart before step 1 completes.  
That is the defining characteristic of a task that needs an agent rather than a chain.

</small>

#### Loop Control Mechanisms  

<small>Without loop control, agents can:   
- Run forever  
- Repeat useless actions  
- Spam APIs   
- Waste tokens  
- Enter failure cycles </nsmall>

**1. Max Iterations**  
<small>Limits total reasoning cycles.  

Example:  
MAX_ITERATIONS = k  
while step_count < MAX_STEPS:
    run_agent_step()
if step_count > MAX_ITERATIONS:  
&emsp;terminate("Task stopped: iteration limit exceeded")  

Purpose:  
- Prevent infinite loops  
- Control cost  
- Ensure predictable runtime</nsmall>

**2. Stopping Conditions**  
<small>Agent should stop when:  
- Goal achieved  
- Tool says “done”  
- Confidence is high  
- No more actions available  
- Repeated failures detected  

Example:   
if state["booking_confirmed"]:  
&emsp;break</nsmall>

### Error Recovery

<small>most beginner implementations:  
try:  
&emsp;tool.run()  
except:  
&emsp;pass  
This is NOT sufficient. 
 
Real agents must:  
- Diagnose failures   
- Retry intelligently  
- Switch strategies  
- Use fallback tools  
- Ask clarifying questions  
- Modify plans dynamically</nsmall>

<small>Types of tool failures
| Failure Type            | Example            |
| ----------------------- | ------------------ |
| Timeout                 | API takes too long |
| Invalid output          | Missing fields     |
| Rate limit              | Too many requests  |
| Tool unavailable        | Service down       |
| Hallucinated tool usage | Wrong parameters   |
| Parsing failure         | JSON malformed     |
| Empty result            | No search results  |
| Authentication error    | Invalid API key    |</nsmall>

<small>**Deliberate Failure**  
User asks: "Schedule a meeting tomorrow at 4 PM."    
Agent uses:  
- Calendar tool  
- Notification tool  

Calendar tool fails with:  
{  
&emsp;"error": "503 Service Unavailable"  
}  
A weak agent crashes.  
A strong agent recovers.</nsmall>

#### Recovery Logic

<small>**Detect Failure**  
result = calendar_tool.run(data)  
if "error" in result:  
&emsp;tool_failed = True</nsmall>

<small>**Failure classification**  
error_type = classify_error(result) 

Classifications:   
- TEMPORARY  
- INVALID_INPUT  
- AUTH_FAILURE  
- EMPTY_RESULT
- UNKNOWN</nsmall>

<small>**Recovery Strategies**  
| Error Type       | Recovery            |
| ---------------- | ------------------- |
| Timeout          | Retry               |
| Rate limit       | Backoff and retry   |
| Invalid input    | Reformat parameters |
| Empty result     | Alternative query   |
| Tool unavailable | Use fallback tool   |
| Auth error       | Escalate to user    |
| Unknown          | Abort safely        |</nsmall>

<small>**Retry Logic**  
Do not just implement: try again  

Instead do: 
- Limit retries  
- Change behavior  
- Add delay  
- Modify parameters  

Example: **exponential backoff**  
retry_count += 1  
if retry_count <= MAX_RETRIES:  
&emsp;wait(2 ** retry_count)</nsmall>

<small>**Fallback Tool Strategy**  
If primary calendar API fails:  
GoogleCalendarTool → OutlookCalendarTool  

Example: **real agent resilience**  
if google_failed:  
&emsp;use_outlook_calendar()

<small>**Re-Planning After Failure**   
The best agents re-plan dynamically. 

Example:  
Thought: Calendar API unavailable.  
Alternative approach: Store reminder locally and notify user.</nsmall>

<small>**Full Recovery Loop Example**

MAX_ITERATIONS = 5  
MAX_RETRIES = 2  

for iteration in range(MAX_ITERATIONS):  
&emsp;thought = agent.reason(state)  
&emsp;action = agent.choose_action(thought)  
&emsp;tool = tools[action["tool"]]  
&emsp;result = tool.run(action["input"])  
&emsp;# Failure handling  
&emsp;if "error" in result:  
&emsp;&emsp;error_type = classify_error(result)  
&emsp;&emsp;if error_type == "TEMPORARY":  
&emsp;&emsp;&emsp;retries += 1  
&emsp;&emsp;&emsp;if retries <= MAX_RETRIES:  
&emsp;&emsp;&emsp;&emsp;sleep(2 ** retries)    
&emsp;&emsp;&emsp;&emsp;continue  
&emsp;&emsp;elif error_type == "INVALID_INPUT":  
&emsp;&emsp;&emsp;action["input"] = repair_input(action["input"])   
&emsp;&emsp;&emsp;continue  
&emsp;&emsp;elif error_type == "TOOL_UNAVAILABLE":   
&emsp;&emsp;&emsp;tool = fallback_tools[action["tool"]]  
&emsp;&emsp;&emsp;result = tool.run(action["input"])  
&emsp;&emsp;else:  
&emsp;&emsp;&emsp;return "Task failed safely"  
&emsp;state.update(result)  
&emsp;if state["goal_completed"]:  
&emsp;&emsp;break   
</nsmall>

<small>**Agentic Recovery**  
detect_failure()  
classify_failure()  
choose_recovery_strategy()  
retry_or_replan()  
continue_execution()  

**Agent Structure**  
state = {  
&emsp;"goal": "...",  
&emsp;"completed_steps": [],  
&emsp;"failed_steps": [],  
&emsp;"tool_history": [],   
&emsp;"retry_counts": {},  
&emsp;"observations": [],  
&emsp;"current_plan": []  
}

**Structured Tool Output**  
{  
&emsp;"status": "success",  
&emsp;"data": {...}  
}  

**Tool History Tracking**  
tool_history.append({  
&emsp;"tool": "calendar",  
&emsp;"input": data,  
&emsp;"result": result  
})</nsmall>

### Agent Safety 

**Permission boundaries :**  
<small>They define what an agent is allowed to access or do.  

Without boundaries, an agent could:  
- delete files,  
- send emails,  
- execute dangerous commands,  
- leak confidential data,  
- make unauthorized purchases.</nsmall>

**Principle of Least Privilege :**  
<small>An agent should only receive:    
- the minimum tools,  
- minimum data,  
- minimum permissions needed for the task.  

Example:  
A calendar scheduling agent:  
- Can read calendar events  
- Can create meetings  
- Cannot access banking apps  
- Cannot execute shell commands</nsmall>

<small>Tool Access Categories
| Tool Type                | Access Level   | Example                             |
| ------------------------ | -------------- | ----------------------------------- |
| Read-only tools          | Safer          | Search API, documentation retrieval |
| Write tools              | Medium risk    | Email sender, database update       |
| Execution tools          | High risk      | Shell commands, Python execution    |
| Financial/critical tools | Very high risk | Payment systems, cloud deployment   |</nsmall>

**Tool Whitelisting :**  
<small>Agents should use only explicitly approved tools.  

ALLOWED_TOOLS = [  
&emsp;"search_docs",  
&emsp;"calendar_api",  
&emsp;"weather_api"  
]  
if requested_tool not in ALLOWED_TOOLS:  
&emsp;raise PermissionError("Tool access denied")</nsmall>

**Sandboxing :**  
<small>Running agents in isolated environments like:  
- containers,  
- virtual machines,  
- restricted APIs,  
- temporary credentials.  

This prevents system-wide damage if the agent behaves unexpectedly.</nsmall>

#### Loop Limits and Runaway Prevention

**Timeout Limits**  
<small>Prevent excessive execution time.  

import time  
start = time.time()  
if time.time() - start > Threshold:  
&emsp;stop_agent("Execution timeout")</nsmall>

**Duplicate Action Detection**  
<small>Prevent repeating same failed action.  

Example:  
if current_action in previous_actions:  
&emsp;repeated_count += 1  
if repeated_count > threshold:  
&emsp;stop_agent("Repeated action detected")</nsmall>

#### Approval steps for high-stakes Actions

<small>High-impact actions should require human approval before execution.  
This is known as Human-in-the-loop (HITL), confirmation gating or approval workflows.  

Examples:  
Sending external emails  
Deploying production code  
Financial transactions  
Deleting databases  
Modifying legal documents  
Accessing private records</nsmall>

**Approval Workflow** 

<small>**Agent Plans Action**  
Agent wants to: "Delete 500 inactive customer records"  

**Summary**  
Proposed Action:   
- Delete 500 records  
- Database: customers  
- Reason: inactive > 2 years  

**Human Approval**  
Approve? (yes/no)  

**Execute Only After Approval**  
if human_approved:  
&emsp;execute_action()  
else:   
&emsp;cancel()</nsmall>

**Logging** 
<small> 
| Event                   | Example                        |
| ----------------------- | ------------------------------ |
| User request            | "Book a meeting tomorrow"      |
| Agent reasoning summary | "Need calendar access"         |
| Tool calls              | `calendar.create_event()`      |
| Tool outputs            | "Meeting created successfully" |
| Errors                  | "API timeout"                  |
| Approval requests       | "Awaiting user confirmation"   |
| Final actions           | "Email sent"                   |

Structured Logging Example:  
{
&ensp;"timestamp": "2026-05-18T10:30:00",  
&emsp;"agent": "scheduler_agent",  
&emsp;"step": 4,  
&emsp;"action": "create_calendar_event",  
&emsp;"status": "success"  
}

Do NOT log:  
- hidden chain-of-thought,  
- sensitive credentials,  
- personal secrets,  
- raw authentication tokens  

Instead log:  
- concise reasoning summaries  
- action intents  
- decision outcomes</nsmall>

**Tool Failure Scenario**  
<small>An agent tries to email a report.  
email_api.send() → SMTP timeout  

**Unsafe Behavior**  
- Retry forever  
- Retry every second  
- Spam the API  

**Escalation**  
After repeated failures:  
- notify administrator
- log the issue
- pause execution
- request human intervention.